# ЛР №9. Теоретические сведения

**Тема:** Развертывание распределенного приложения в Kubernetes

## 1. Зачем нужен Kubernetes

В ЛР №8 приложение упаковано в Docker-образ и запускается в контейнере. По мере роста системы возникают задачи, которые Docker сам по себе не решает:

- запуск нескольких экземпляров одного сервиса;
- поддержание заданного количества экземпляров;
- стабильный доступ к приложению при пересоздании экземпляров;
- восстановление экземпляров после сбоев;
- централизованное описание конфигурации;
- управление большим числом контейнеризированных компонентов.

**Kubernetes** — платформа оркестрации контейнеризированных приложений. Ключевая идея: разработчик описывает **желаемое состояние** системы (например, «три экземпляра сервера»), а Kubernetes постоянно сравнивает фактическое состояние с желаемым и устраняет расхождения.

## 2. Kubernetes и Docker: разные задачи

| Технология | Основное назначение |
|---|---|
| Docker | Создание образов и запуск контейнеров |
| Dockerfile | Описание процесса сборки Docker-образа |
| Docker Compose | Совместное описание и локальный запуск нескольких сервисов на одном хосте |
| Kubernetes | Управление развертыванием, экземплярами и состоянием контейнеризированных приложений в кластере |
| kubectl | Взаимодействие с кластером Kubernetes из командной строки |

Docker отвечает за **что запускать** (образ и контейнер). Kubernetes отвечает за **как и сколько запускать и поддерживать** (экземпляры, доступ, восстановление, конфигурацию). Kubernetes использует образы Docker (или совместимые с OCI) и не заменяет их.

## 3. Основные понятия

### 3.1. Cluster (кластер)

Среда, в которой размещаются и управляются приложения. Состоит из управляющих компонентов (control plane) и узлов, на которых выполняются рабочие нагрузки.

Основные команды:

```bash
kubectl cluster-info     # информация о кластере
kubectl get nodes        # список узлов
```

В учебной работе инфраструктура скрыта за Docker Desktop или Minikube; достаточно уметь проверить доступность кластера и его состояние.

### 3.2. Node (узел)

Машина (физическая или виртуальная), на которой выполняются Pod. В реальном кластере узлов много; локально обычно один.

Важно различать **количество узлов** и **количество экземпляров приложения**: на одном узле может работать три Pod сервера.

```bash
kubectl get nodes
kubectl describe node <имя-узла>
```

### 3.3. Pod

Минимальная единица развертывания в Kubernetes. Pod содержит один или несколько тесно связанных контейнеров, которые используют общую сеть и общие ресурсы. В ЛР один Pod содержит один контейнер серверного приложения.

Особенности Pod:
- получает собственный IP-адрес внутри кластера;
- адрес **не постоянен**: при пересоздании Pod адрес меняется;
- Pod эфемерен: он создается, работает и удаляется; данные, записанные внутри него, при удалении теряются.

Состояния Pod:

| Состояние | Значение |
|---|---|
| Pending | Pod еще не запущен, например, ожидает ресурсы или загрузку образа |
| Running | Pod привязан к узлу, контейнеры созданы; это не означает, что приложение готово принимать запросы |
| Succeeded | Выполнение завершилось успешно (характерно для задач, рассчитанных на завершение) |
| Failed | Выполнение завершилось с ошибкой |
| CrashLoopBackOff | Контейнер неоднократно падает, Kubernetes увеличивает паузы между перезапусками |
| Unknown | Состояние неизвестно (обычно потеряна связь с узлом) |

Диагностика:

```bash
kubectl get pods
kubectl describe pod <имя-pod>    # события, причина, образ, условия
kubectl logs <имя-pod>            # журнал контейнера
```

Для диагностики важно смотреть не только название статуса, но и события, журналы и причину завершения контейнера.

### 3.4. ReplicaSet

Контроллер, который поддерживает заданное число одинаковых Pod. Обычно не создается вручную: его создает и обновляет Deployment.

### 3.5. Deployment

Ресурс, который описывает желаемое состояние развертывания приложения и управляет ReplicaSet и Pod.

В Deployment задаются:
- имя приложения;
- Docker-образ;
- количество экземпляров (`replicas`);
- метки Pod (`labels`);
- параметры запуска контейнера;
- используемые порты;
- переменные окружения;
- правила обновления.

Пример:

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: riot-server
spec:
  replicas: 3
  selector:
    matchLabels:
      app: riot-server
  template:
    metadata:
      labels:
        app: riot-server
    spec:
      containers:
        - name: server
          image: riot-server:1.0
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: 8080
```

Если удалить один из Pod, Deployment через ReplicaSet обнаружит отклонение от `replicas: 3` и создаст замену.

Ограничение: Deployment не гарантирует, что все экземпляры будут работать при любых обстоятельствах. Восстановление зависит от доступности узлов, ресурсов, образов и корректности конфигурации.

### 3.6. Scaling (масштабирование)

Изменение количества экземпляров (Pod):

```bash
kubectl scale deployment riot-server --replicas=3
kubectl get deployments
kubectl get pods
```

Масштабирование не ускоряет приложение автоматически. Результат зависит от:
- характера нагрузки;
- ресурсов узла (экземпляры делят CPU и память);
- архитектуры сервера;
- общих данных и их синхронизации;
- доступности зависимых компонентов (например, БД).

Если несколько экземпляров одновременно меняют общие данные, приложение должно обеспечивать корректность: транзакции, блокировки, согласованность. Увеличение числа Pod этого не заменяет.

### 3.7. Service

Ресурс, который предоставляет **стабильную точку доступа** к группе Pod: единый логический адрес и порт, через который трафик направляется на подходящие Pod.

Без Service клиенту пришлось бы самому знать адреса всех Pod и выбирать, какой из них жив. Service скрывает эти детали.

Основные параметры:

| Параметр | Назначение |
|---|---|
| `port` | Порт, на котором доступен Service |
| `targetPort` | Порт контейнера, на который направляется трафик |
| `selector` | Набор меток, по которым Service выбирает Pod |
| `type` | Способ предоставления доступа (например, ClusterIP) |
| `protocol` | Сетевой протокол (TCP) |

Типы Service:
- **ClusterIP** — доступ только изнутри кластера; для доступа с компьютера используется `port-forward`;
- **NodePort**, **LoadBalancer** — публикация наружу (в ЛР не требуются).

Service не исправляет ошибки приложения и не гарантирует доступность внешней БД: он лишь направляет трафик на подходящие Pod.

```bash
kubectl get services
kubectl describe service riot-server
```

### 3.8. Label и Selector

**Label** — пара «ключ-значение», назначаемая объекту:

```yaml
labels:
  app: riot-server
```

**Selector** — правило выбора объектов по меткам.

- Deployment использует селектор, чтобы найти свои Pod;
- Service использует селектор, чтобы найти Pod, которым направлять запросы.

Метки в Deployment и селектор Service должны совпадать. Если Service ищет `app: riot-server`, а Pod имеет только `app: server`, Service не выберет Pod. Приложение при этом может быть запущено, но недоступно через Service — это одна из самых частых ошибок.

Проверка:

```bash
kubectl get pods --show-labels
kubectl describe service riot-server     # раздел Selector
```

### 3.9. YAML

Текстовый формат структурированных данных с отступами. В Kubernetes им описываются ресурсы. Обязательные поля верхнего уровня:

- `apiVersion` — версия API ресурса;
- `kind` — тип ресурса (Deployment, Service, ConfigMap, Pod);
- `metadata` — имя и метаданные (метки, аннотации);
- `spec` — желаемое состояние ресурса (для ConfigMap вместо `spec` используется `data`).

Применение и удаление:

```bash
kubectl apply -f deployment.yaml     # создать или обновить ресурс
kubectl delete -f deployment.yaml    # удалить ресурсы из файла
```

Kubernetes принимает описание и стремится привести фактическое состояние к указанному. Команда `kubectl apply` подтверждает **применение** конфигурации, но не гарантирует, что приложение успешно запустилось.

### 3.10. ConfigMap и переменные окружения

**ConfigMap** — ресурс для хранения **неконфиденциальных** параметров конфигурации: режима работы, названия среды, параметров поведения, адреса зависимого сервиса.

```yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: riot-config
data:
  APP_MODE: "kubernetes"
```

Параметр передается в контейнер через **переменную окружения** — параметр, доступный процессу во время выполнения:

```yaml
env:
  - name: APP_MODE
    valueFrom:
      configMapKeyRef:
        name: riot-config
        key: APP_MODE
```

В C# значение читается так:

```csharp
string mode =
    Environment.GetEnvironmentVariable("APP_MODE")
    ?? "default";
```

Важные правила:
- `name` в `configMapKeyRef` должен совпадать с именем ConfigMap, `key` — с ключом в `data`;
- переменная окружения уже запущенного процесса **не меняется автоматически** при изменении ConfigMap; нужно пересоздать Pod (например, `kubectl rollout restart deployment <имя>`);
- пароли и секреты в ConfigMap не хранятся; для них предназначен Secret.

### 3.11. Порт контейнера: `containerPort`

Поле `containerPort` документирует порт, который использует приложение. Само по себе оно **не меняет** настройки приложения и **не публикует** порт на компьютере пользователя.

### 3.12. Доступ с локального компьютера: `kubectl port-forward`

```bash
kubectl port-forward service/riot-server 8080:8080
```

Команда перенаправляет локальный порт 8080 на порт 8080 Service. Пока команда работает, клиент на том же компьютере обращается к серверу через `localhost:8080`. Команду обычно оставляют работать в отдельном терминале; после ее завершения перенаправление прекращается.

`localhost` здесь используется **клиентом на компьютере пользователя**. Внутри кластера сервер не должен обращаться к другим сервисам через `localhost` — для этого используются имена Service.

### 3.13. Журналы и события

```bash
kubectl logs <имя-pod>                               # журнал контейнера
kubectl get events --sort-by=.metadata.creationTimestamp   # последние события кластера
```

Для просмотра журналов нужно сначала узнать фактическое имя Pod через `kubectl get pods`.

## 4. Порядок диагностики

Если приложение недоступно, проверяется последовательно:

1. Доступен ли кластер (`kubectl cluster-info`, `kubectl get nodes`).
2. Не обращается ли kubectl к другому контексту (`kubectl config current-context`).
3. Создан ли Deployment (`kubectl get deployments`).
4. Созданы ли Pod (`kubectl get pods`).
5. В каком состоянии находятся Pod.
6. Успешно ли запускается приложение (`kubectl logs`).
7. Совпадают ли метки Pod и селектор Service.
8. Совпадают ли `port` и `targetPort` с реальной конфигурацией.
9. Получает ли приложение нужные параметры (переменные окружения).
10. Нет ли ошибок в журналах и событиях.

## 5. Восстановление после отказа

Kubernetes описывает желаемое состояние, а контроллеры отслеживают управляемые ресурсы и восстанавливают их. При удалении Pod, управляемого Deployment:

1. Pod завершается.
2. Deployment/ReplicaSet видит, что фактических Pod меньше `replicas`.
3. Создается новый Pod (с новым именем и адресом).
4. Новый Pod должен пройти запуск: загрузка образа, старт контейнера, готовность приложения.

Важно различать разные события, которые не равнозначны:
- удаление Pod и завершение процесса приложения;
- перезапуск контейнера внутри Pod;
- пересоздание Pod;
- восстановление доступности приложения;
- восстановление данных;
- восстановление внешних сервисов.

Например, Kubernetes может создать новый Pod, но приложение не сможет работать, если недоступна БД. Если данные хранились только внутри удаленного Pod, Kubernetes их не восстановит.

## 6. Особенности локального кластера

- Несколько Pod могут выполняться на одном физическом компьютере.
- Отказ компьютера приводит к недоступности всего локального кластера.
- Несколько экземпляров не гарантируют независимость от общей БД.
- Локальное масштабирование не всегда повышает производительность.
- Доступность зависит от корректности конфигурации и состояния зависимых компонентов.

## 7. Сравнение Docker Compose и Kubernetes

| Критерий | Docker / Docker Compose | Kubernetes |
|---|---|---|
| Основная задача | Запуск сервисов на одном хосте | Поддержание желаемого состояния в кластере |
| Описание конфигурации | `Dockerfile`, `compose.yaml` | YAML-ресурсы (Deployment, Service, ConfigMap) |
| Запуск | `docker compose up` | `kubectl apply -f ...` |
| Количество экземпляров | Вручную (`--scale` для compose-сервисов) | `replicas`, `kubectl scale` |
| Доступ | Публикация порта, DNS по именам сервисов | Service с селектором по меткам |
| Реакция на удаление контейнера/Pod | `restart: unless-stopped` на том же хосте | Контроллер создает новый Pod |
| Диагностика | `docker logs`, `docker compose ps` | `kubectl describe`, `logs`, `get events` |
| Среда | Docker Engine | Кластер (Docker Desktop, Minikube), kubectl, доступный кластеру образ |

## 8. Ошибки, которые нужно уметь диагностировать

| Ситуация | Признак | Что проверить |
|---|---|---|
| Кластер недоступен | kubectl не подключается | `kubectl cluster-info`, запущен ли Docker Desktop / Minikube |
| Неверный контекст | Ресурсы «не видны» | `kubectl config current-context` |
| Образ отсутствует в кластере | ErrImagePull / ImagePullBackOff | Загружен ли образ (`minikube image load`), имя и тег |
| Неверный тег образа | Те же статусы | `image:` в Deployment и `docker images` |
| Pod не запускается | Pending, CrashLoopBackOff | `kubectl describe pod`, `kubectl logs`, события |
| Приложение падает | CrashLoopBackOff, ошибки в логах | Журнал контейнера, переменные окружения |
| Service не выбирает Pod | Нет ответа через Service | Совпадение меток и селектора |
| Неверный port/targetPort | Соединение отклоняется | Порт, который слушает приложение, и значения в Service |
| port-forward не работает | Клиент не подключается | Запущена ли команда, занят ли локальный порт |
| ConfigMap не применен | Приложение использует старое/дефолтное значение | Имя ConfigMap, ключ, пересоздание Pod |
| Новый Pod не запускается после удаления | Pod не Running | `kubectl describe`, журнал, события |
| Зависимый компонент недоступен | Приложение запущено, но операции падают | Доступность БД и других сервисов |

Общий алгоритм: зафиксировать состояние → получить `describe` → посмотреть события и журналы → определить причину → исправить конфигурацию → повторно проверить состояние и выполнить запрос.

## 9. Контрольные вопросы (краткие ответы)

1. **Что такое Kubernetes и для чего он используется?** Платформа оркестрации контейнеров: поддерживает желаемое состояние приложения (экземпляры, доступ, конфигурация, восстановление).
2. **Чем Kubernetes отличается от Docker?** Docker создает образы и запускает контейнеры на одном хосте; Kubernetes управляет контейнерами в кластере и поддерживает заданное состояние.
3. **Что такое Kubernetes Cluster?** Среда с управляющими компонентами и узлами, на которых выполняются Pod.
4. **Что такое Pod?** Минимальная единица развертывания: один или несколько тесно связанных контейнеров с общей сетью.
5. **Чем Pod отличается от Docker-контейнера?** Pod — обертка Kubernetes над контейнерами, со своим IP-адресом и жизненным циклом.
6. **Что такое Deployment?** Ресурс, описывающий желаемое состояние развертывания и управляющий ReplicaSet и Pod.
7. **Для чего используется `replicas`?** Задает желаемое количество Pod.
8. **Что такое Service?** Стабильная точка доступа к группе Pod.
9. **Почему клиенту неудобно обращаться к Pod напрямую?** Адреса Pod меняются при пересоздании, их несколько, и клиент не знает, какой из них доступен.
10. **Для чего используются метки и селекторы?** Метки помечают объекты; селекторы выбирают их. Deployment находит свои Pod, Service — целевые Pod.
11. **Что такое YAML и зачем он нужен в Kubernetes?** Формат описания ресурсов и их желаемого состояния.
12. **Как проверить состояние Pod?** `kubectl get pods`, `kubectl describe pod`, `kubectl logs`.
13. **Как получить подробную информацию о Pod?** `kubectl describe pod <имя>`.
14. **Что происходит при удалении Pod, управляемого Deployment?** Deployment создает новый Pod, чтобы вернуть заданное количество.
15. **Почему восстановление Pod не гарантирует сохранность данных?** Данные внутри Pod теряются при его удалении; новый Pod стартует с чистого состояния.
16. **Что такое ConfigMap?** Ресурс для хранения неконфиденциальных параметров конфигурации.
17. **Для чего используются переменные окружения?** Передача параметров процессу без изменения кода и образа.
18. **Для чего используется `kubectl port-forward`?** Перенаправление локального порта на Service для доступа с компьютера.
19. **Почему несколько Pod на одном компьютере не защищают от отказа этого компьютера?** Все Pod зависят от одного физического узла.
20. **Чем Docker Compose отличается от Kubernetes?** Compose запускает сервисы на одном хосте; Kubernetes управляет экземплярами в кластере и восстанавливает их.

## 10. Краткие выводы

- Kubernetes описывает **желаемое** состояние приложения и поддерживает его.
- Deployment отвечает за экземпляры, Service — за стабильный доступ, ConfigMap — за конфигурацию.
- Метки и селекторы связывают все ресурсы; их несоответствие — частая причина недоступности.
- Успешный `kubectl apply` не означает работающее приложение: нужно проверить Pod, журналы и запрос.
- Восстановление количества Pod не равно восстановлению данных и внешних сервисов.